# GS-001 — Inline Companion & Execution Verification Notebook

**AuthorityEffect: None.** Mechanizes and executes all inline verification blocks for GS-001.

- **Theorem 4.2:** Maximal End_b(V) local symmetry forces local native-norm rigidity.
- **CPTP Rigidity:** Measure-and-prepare channel contractivity on H_n^0 forces Schatten-1 trace norm.
- **Convex Duality:** Unit ball scaling B_j = c^{-1} B_2^k.


In [1]:
import sys
from pathlib import Path
import numpy as np

def _boot_libs(*names):
    """Robustly resolve and import library directories regardless of CWD."""
    cwd = Path.cwd().resolve()
    search_dirs = [cwd] + list(cwd.parents)
    for ancestor in [cwd] + list(cwd.parents):
        cand = ancestor / "coda_atlas" / "F004_papers" / "002_projection_obstruction" / "007_m_compatible_gauge_classification"
        if cand.exists():
            search_dirs.insert(0, cand)
            break
    for name in names:
        for base in search_dirs:
            p = (base / name).resolve()
            if p.is_dir() and p.exists():
                if str(p) not in sys.path:
                    sys.path.insert(0, str(p))
                return str(p)
            sub = base / "007_m_compatible_gauge_classification" / name
            if sub.is_dir() and sub.exists():
                sub_res = sub.resolve()
                if str(sub_res) not in sys.path:
                    sys.path.insert(0, str(sub_res))
                return str(sub_res)
    raise FileNotFoundError(f"none of {names} found from {cwd}")

_boot_libs("code")
from gs001_gauges import (
    l1, l2, linf, anisotropic_gauge, exact_induced_operator_norm,
    minkowski_functional, unit_ball_scaling, rank_one_map,
    construct_hahn_banach_rank_one, is_maximal_end_compatible,
    cptp_measure_and_prepare, is_cptp_trace_norm_rigid, trace_norm
)
print("GS-001 Verification Companion Engine Initialized")

GS-001 Verification Companion Engine Initialized


## Block 1: Theorem 3.1 & Convex Unit Ball Scaling Verification

Verify that for $j(x) = c \|x\|_2$, the unit ball $B_j = \{x : j(x) \le 1\}$ has radius $c^{-1}$ ($B_j = c^{-1} B_2^k$).


In [2]:
res = unit_ball_scaling(c=2.5, dimension=3)
print(f"Scaling c=2.5 -> Unit ball radius = {res['radius_of_unit_ball']:.4f} (Formula: {res['formula']})")
assert np.isclose(res['radius_of_unit_ball'], 0.4)

x_test = np.array([3.0, -4.0])
j_mink = minkowski_functional(x_test, lambda v: l2(v) <= 1.0)
print(f"Minkowski round trip j_B_j(x) = {j_mink:.6f} (Exact l2 = {l2(x_test)})")
assert np.isclose(j_mink, 5.0, atol=1e-5)


Scaling c=2.5 -> Unit ball radius = 0.4000 (Formula: B_j = (1/2.5) * B_2^3)
Minkowski round trip j_B_j(x) = 5.000000 (Exact l2 = 5.0)


## Block 2: Hahn-Banach Rank-One Rigidity Verification

Verify $T = (v \otimes y^T) / y(u)$ satisfies $T u = v$ for any non-zero source vector $u$.

In [3]:
u = np.array([2.0, 0.0])
v = np.array([0.0, 1.0])
T = rank_one_map(u, v)
print(f"Rank-one contract T @ u = {T @ u} (Target v = {v})")
assert np.allclose(T @ u, v)

u_hb = np.array([3.0, 4.0])
v_hb = np.array([1.0, -2.0])
T_hb = construct_hahn_banach_rank_one(u_hb, v_hb, l2, "l2")
print(f"Hahn-Banach rank-one T_hb @ u = {T_hb @ u_hb} (Target v = {v_hb})")
assert np.allclose(T_hb @ u_hb, v_hb)


Rank-one contract T @ u = [0. 1.] (Target v = [0. 1.])
Hahn-Banach rank-one T_hb @ u = [ 1. -2.] (Target v = [ 1. -2.])


## Block 3: CPTP Measure-and-Prepare Trace-Norm Rigidity Verification

Verify measure-and-prepare CPTP quantum channels on $H_n^0$ strictly force Schatten-1 trace norm.

In [4]:
X = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
Y = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)

ch, kraus = cptp_measure_and_prepare(X, Y)
LX = ch(X)
print(f"CPTP Measure-and-Prepare Lambda(X) == Y: {np.allclose(LX, Y)}")
assert np.allclose(LX, Y)

is_rigid, j_X, j_Y = is_cptp_trace_norm_rigid(X, Y, trace_norm)
print(f"Trace Norm CPTP Rigidity Verified: {is_rigid} (j(X)={j_X:.4f}, j(Y)={j_Y:.4f})")
assert is_rigid


CPTP Measure-and-Prepare Lambda(X) == Y: True
Trace Norm CPTP Rigidity Verified: True (j(X)=2.0000, j(Y)=2.0000)
